In [1]:
import numpy as np
import pandas as pd

from datetime import timedelta

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

import warnings
warnings.filterwarnings('ignore')

In [2]:
dc = pd.read_csv('.\data\customers.csv')
dc['date_of_birth'] = pd.to_datetime(dc['date_of_birth'])
dc['created_at'] = pd.to_datetime(dc['created_at']).dt.floor('d')
dc

,id,first_name,last_name,email,phone,city,date_of_birth,created_at
0,1,Wesley,Cline,jarvissharon@example.com,(341)849-2772x5766,Stuartland,1988-08-11,2024-11-22
1,2,Jennifer,Fowler,jeremiah48@example.org,(488)331-4684x1733,Romanfort,1961-03-12,2024-11-26
2,3,Joseph,Bell,courtney35@example.com,+1-223-732-9514,North Robert,1981-03-22,2024-03-16
3,4,Toni,Barker,perezfernando@example.com,373-220-8928,Davidhaven,1957-02-10,2024-08-15
4,5,William,Mccarthy,tparker@example.com,987.385.2246,North Stephenfort,1975-01-10,2024-11-16
...,...,...,...,...,...,...,...,...
995,996,Lori,Joyce,gomezmiguel@example.net,397-521-0405,Courtneyport,1951-09-28,2024-05-09
996,997,Jon,Ward,rogersrobert@example.net,(967)526-7863,Rhondaton,1981-01-22,2024-05-29
997,998,Robert,Barrera,kcabrera@example.org,+1-360-917-6510x5895,South Allenland,1945-07-14,2024-09-12
998,999,Lori,Bailey,victoria06@example.com,9432414621,Port Albertport,1976-04-13,2024-01-21


In [3]:
do = pd.read_csv('.\data\orders.csv')
do['order_date'] = pd.to_datetime(do['order_date']).dt.floor('d')
do

,order_id,customer_id,product_id,order_date,total_amount
0,1,603,30,2024-09-22,1914.07
1,2,695,3,2024-06-22,2702.46
2,3,567,24,2024-11-04,165.93
3,4,760,38,2024-09-06,3426.40
4,5,716,23,2024-08-07,3352.38
...,...,...,...,...,...
4995,4996,163,35,2024-07-06,2272.80
4996,4997,618,29,2024-08-20,1775.98
4997,4998,841,18,2024-07-28,1274.74
4998,4999,628,22,2024-08-05,783.38


In [ ]:
data = pd.merge(dc, do, left_on = 'id', right_on='customer_id', how='inner')

In [ ]:
# сначала купил, потом зарегистрировался?
data[data['created_at'] > data['order_date']].shape[0]

In [ ]:
created = data[data['created_at'] > data['order_date']].groupby('customer_id')['order_date'].transform('min')
created.name = 'created_at'
# created

In [ ]:
dc.set_index(['id'], inplace=True)
dc.update(created)
dc.reset_index(inplace=True)

In [ ]:
# дата регистрации = дата первой покупки
data.loc[data['created_at'] > data['order_date'], 'created_at'] = created

In [ ]:
data['year'] = data['order_date'].dt.year
data['month'] = data['order_date'].dt.month

In [ ]:
# первый день периода
data['first_day'] = pd.to_datetime(data['year'].astype(str) + '-' + data['month'].astype(str).str.zfill(2) + '-01')

`R` – количество дней, прошедших с момента последней покупки до начала следующего периода\
`F` – количество заказов в пероде\
`M` – общая сумма затрат клиента в периоде\
`life_time` - количество дней от даты регистрации до конца периода\
`age` - возраст на начало периода

In [ ]:
def age(birth_date, today):
    age = today.year - birth_date.year
    if all([x >= y for x,y in zip(today.timetuple(), birth_date.timetuple())]):
        return age
    return age - 1

In [ ]:
df = pd.DataFrame()
for m in np.sort(data['month'].unique()):
    first_day = pd.to_datetime('2024-' + str(m).zfill(2) + '-01')
    last_day = pd.to_datetime('2024-' + str(m+1).zfill(2) + '-01') - timedelta(days=1)
    #print(first_day, last_day)
    clients = dc[dc['created_at']<=last_day]['id']
    df = pd.concat([df, pd.DataFrame({'customer_id': clients,
                                      'month': m,
                                      'age': dc[dc['id'].isin(clients)]['date_of_birth'].apply(lambda x: age(x,first_day)),
                                      'life_time': (last_day - dc[dc['id'].isin(clients)]['created_at']).dt.days+1,
                                      'R': 0, 'F': 0, 'M': 0})], ignore_index=True)

In [ ]:
# последний заказ в периоде
gr = data.groupby(['customer_id','month','first_day'], as_index=False).agg(
    last_order = ('order_date', 'max'),
    F = ('order_id', 'count'),
    M = ('total_amount', 'sum')
)
gr['R'] = (gr['first_day'] - gr.groupby('customer_id')['last_order'].shift(1)).dt.days
gr['R'].fillna(-1, inplace=True)
# gr

In [ ]:
df.set_index(['customer_id','month'], inplace=True)
df.update(gr.set_index(['customer_id','month']))
df.reset_index(inplace=True)

In [ ]:
df['Y'] = df['F'].apply(lambda x: int(x>0))

In [ ]:
df.to_csv('.\data\dataset.csv', sep=';', index=False)